# Ordenación por burbuja (bubble sort)

Recorre la lista intercambiando pares adyacentes desordenados; en cada pasada el mayor "sube" al final. Cuesta $O(n^2)$, pero es el algoritmo de ordenación más fácil de entender.

## 1. Burbuja básica

Ordenar una lista de menor a mayor intercambiando elementos adyacentes.

In [1]:
def bubble_sort(lista):
    l = lista[:]
    for i in range(len(l)):
        for j in range(len(l) - 1):
            if l[j] > l[j + 1]:
                l[j], l[j + 1] = l[j + 1], l[j]
    return l

print(bubble_sort([5, 1, 4, 2, 8, 0]))

[0, 1, 2, 4, 5, 8]


## 2. Burbuja optimizada y descendente

Optimizar el algoritmo para no repetir trabajo: tras la pasada $i$, los últimos $i$ elementos ya están colocados, y si una pasada no intercambia nada la lista ya está ordenada. Añadir la opción de ordenar de mayor a menor.

In [2]:
import random

def bubble_sort(lista, descendente=False):
    l = lista[:]
    for i in range(len(l)):
        intercambio = False
        # los últimos i ya están en su sitio
        for j in range(len(l) - 1 - i):
            if (l[j] < l[j + 1]) if descendente else (l[j] > l[j + 1]):
                l[j], l[j + 1] = l[j + 1], l[j]
                intercambio = True
        if not intercambio:
            break
    return l

for _ in range(200):
    datos = random.choices(range(100), k=random.randint(0, 30))
    assert bubble_sort(datos) == sorted(datos)
    assert bubble_sort(datos, descendente=True) == sorted(datos, reverse=True)
print(bubble_sort([5, 1, 4, 2, 8, 0], descendente=True))

[8, 5, 4, 2, 1, 0]


## 3. Comparaciones, intercambios, mejor y peor caso

Contar comparaciones e intercambios de la burbuja optimizada. En el mejor caso (lista ya ordenada) hace $n - 1$ comparaciones y ningún intercambio: $O(n)$. En el peor (orden inverso), $\frac{n(n-1)}{2}$ de cada. Comprobar además que cada intercambio corrige exactamente una inversión (par $i < j$ con $l_i > l_j$), así que hay tantos intercambios como inversiones.

In [3]:
def bubble_sort_contando(lista):
    l, comparaciones, intercambios = lista[:], 0, 0
    for i in range(len(l)):
        intercambio = False
        for j in range(len(l) - 1 - i):
            comparaciones += 1
            if l[j] > l[j + 1]:
                l[j], l[j + 1] = l[j + 1], l[j]
                intercambios += 1
                intercambio = True
        if not intercambio:
            break
    return l, comparaciones, intercambios

def inversiones(lista):
    return sum(lista[i] > lista[j] for i in range(len(lista)) for j in range(i + 1, len(lista)))

random.seed(3)
n = 100
casos = {"ordenada": list(range(n)), "inversa": list(range(n, 0, -1)), "aleatoria": random.sample(range(n), n)}
print(f"{'caso':>10} {'comparaciones':>14} {'intercambios':>13} {'inversiones':>12}")
for nombre, datos in casos.items():
    _, comparaciones, intercambios = bubble_sort_contando(datos)
    print(f"{nombre:>10} {comparaciones:>14} {intercambios:>13} {inversiones(datos):>12}")

assert bubble_sort_contando(casos["ordenada"])[1:] == (n - 1, 0)
assert bubble_sort_contando(casos["inversa"])[1:] == (n * (n - 1) // 2, n * (n - 1) // 2)
for _ in range(200):
    datos = random.choices(range(50), k=random.randint(0, 30))
    assert bubble_sort_contando(datos)[2] == inversiones(datos)

      caso  comparaciones  intercambios  inversiones
  ordenada             99             0            0
   inversa           4950          4950         4950
 aleatoria           4935          2734         2734


## 4. Burbuja bidireccional y estabilidad

Implementar la burbuja bidireccional (*cocktail shaker sort*), que alterna una pasada hacia la derecha y otra hacia la izquierda, con un parámetro `clave`. Comprobar que es estable comparando con `sorted` (que lo es) y romper la estabilidad de la burbuja cambiando `>` por `>=`. Medir la ventaja de ir en los dos sentidos con una «tortuga»: un elemento pequeño al final, que la burbuja normal solo acerca una posición por pasada.

In [4]:
def cocktail_sort(lista, clave=lambda x: x):
    l, comparaciones = lista[:], 0
    izq, der, intercambio = 0, len(l) - 1, True
    while intercambio:
        intercambio = False
        for j in range(izq, der):
            comparaciones += 1
            if clave(l[j]) > clave(l[j + 1]):
                l[j], l[j + 1] = l[j + 1], l[j]
                intercambio = True
        der -= 1
        for j in range(der, izq, -1):
            comparaciones += 1
            if clave(l[j - 1]) > clave(l[j]):
                l[j - 1], l[j] = l[j], l[j - 1]
                intercambio = True
        izq += 1
    return l, comparaciones

def burbuja_con_igual(lista, clave):
    l = lista[:]
    for i in range(len(l)):
        for j in range(len(l) - 1 - i):
            # >= intercambia también los empates
            if clave(l[j]) >= clave(l[j + 1]):
                l[j], l[j + 1] = l[j + 1], l[j]
    return l

def nota(alumno):
    return alumno[1]

alumnos = [("Ana", 7), ("Luis", 5), ("Eva", 7), ("Pau", 10), ("Iker", 5), ("Sara", 7)]
print("Estable:  ", cocktail_sort(alumnos, nota)[0])
print("Con >=:   ", burbuja_con_igual(alumnos, nota))
assert cocktail_sort(alumnos, nota)[0] == sorted(alumnos, key=nota)
assert burbuja_con_igual(alumnos, nota) != sorted(alumnos, key=nota)

random.seed(4)
for _ in range(300):
    datos = [(random.randint(0, 5), i) for i in range(random.randint(0, 20))]
    assert cocktail_sort(datos, lambda r: r[0])[0] == sorted(datos, key=lambda r: r[0])

n = 100
tortuga = list(range(2, n + 1)) + [1]
print(f"Tortuga: {bubble_sort_contando(tortuga)[1]} comparaciones con burbuja, {cocktail_sort(tortuga)[1]} con cóctel")

Estable:   [('Luis', 5), ('Iker', 5), ('Ana', 7), ('Eva', 7), ('Sara', 7), ('Pau', 10)]
Con >=:    [('Luis', 5), ('Iker', 5), ('Sara', 7), ('Ana', 7), ('Eva', 7), ('Pau', 10)]
Tortuga: 4950 comparaciones con burbuja, 390 con cóctel
